<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در این مثال ابتدا با Bi-Encoder پنج سند مرتبط را پیدا می‌کنیم و سپس با Cross-Encoder آن‌ها را مجدداً رتبه‌بندی می‌کنیم.

نکته: مدل cross-encoder/ms-marco-MiniLM-L-6-v2 برای داده‌های انگلیسی و وظایف مرتبط با جستجو آموزش دیده است. برای داده‌های فارسی باید مدل Cross-Encoder مناسب زبان فارسی انتخاب و ارزیابی شود. همچنین امتیاز Cross-Encoder لزوماً احتمال ارتباط سند نیست.

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

# 1. HR Knowledge Base
documents = [
    {"id": "HR-001", "text": "Employees receive 26 annual leave days."},
    {"id": "HR-002", "text": "Leave requests must be approved by the manager."},
    {"id": "HR-003", "text": "Employees have supplementary health insurance."},
    {"id": "HR-004", "text": "Monthly salaries are paid at the end of each month."},
    {"id": "HR-005", "text": "Employees can work remotely two days per week."},
    {"id": "HR-006", "text": "Employees can request unpaid leave."},
]

# 2. Initialize Bi-Encoder
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

document_embeddings = bi_encoder.encode(
    [doc["text"] for doc in documents], convert_to_numpy=True, normalize_embeddings=True
)

# 3. Initialize Cross-Encoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


# 4. First-stage Retrieval
def retrieve(query, top_k=5):
    query_embedding = bi_encoder.encode(
        query, convert_to_numpy=True, normalize_embeddings=True
    )

    scores = document_embeddings @ query_embedding

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in ranked_indices:
        results.append({**documents[index], "retrieval_score": float(scores[index])})

    return results


# 5. Second-stage Reranking
def rerank(query, candidates, top_k=3):
    pairs = [(query, document["text"]) for document in candidates]

    rerank_scores = reranker.predict(pairs)

    results = []

    for document, score in zip(candidates, rerank_scores):
        results.append({**document, "rerank_score": float(score)})

    results.sort(key=lambda item: item["rerank_score"], reverse=True)

    return results[:top_k]


# 6. Execute Pipeline
query = "How many days off can employees take each year?"

candidates = retrieve(query, top_k=5)
final_results = rerank(query, candidates, top_k=3)

print("Initial Retrieval Results:")
for doc in candidates:
    print(doc["id"], round(doc["retrieval_score"], 4))

print("\nReranked Results:")
for rank, doc in enumerate(final_results, start=1):
    print(f"\nRank: {rank}")
    print("ID:", doc["id"])
    print("Text:", doc["text"])
    print("Rerank Score:", round(doc["rerank_score"], 4))

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3770.14it/s]


Initial Retrieval Results:
HR-001 0.6659
HR-005 0.6036
HR-006 0.4994
HR-004 0.3656
HR-003 0.3117

Reranked Results:

Rank: 1
ID: HR-001
Text: Employees receive 26 annual leave days.
Rerank Score: 5.5094

Rank: 2
ID: HR-005
Text: Employees can work remotely two days per week.
Rerank Score: 0.0621

Rank: 3
ID: HR-006
Text: Employees can request unpaid leave.
Rerank Score: -7.2425
